# CEI Bibliometric Analysis — Universidad de Talca

This notebook compares the research article output of the **Center for Integrative Ecology
(Centro de Ecología Integrativa — CEI)** against seven other academic units of
Universidad de Talca (Chile) using data from the [OpenAlex](https://openalex.org) open
bibliometric API.

## Units compared
| Label | Spanish / English names |
|-------|------------------------|
| **CEI** | Centro de Ecología Integrativa / Center for Integrative Ecology |
| CBSM | Centro de Bioinformática, Simulación y Modelado / Center for Bioinformatics, Simulations and Modelling |
| Fac. Ciencias de la Salud | Facultad de Ciencias de la Salud / Faculty of Health Sciences |
| Inst. Ciencias Biológicas | Instituto de Ciencias Biológicas / Institute of Biological Sciences |
| Fac. Ciencias Agrarias | Facultad de Ciencias Agrarias / Faculty of Agrarian Sciences |
| Inst. Química R. Naturales | Instituto de Química de Recursos Naturales |
| Inst. Matemáticas | Instituto de Matemáticas / Institute of Mathematics |
| Fac. Medicina | Escuela / Facultad de Medicina / School of Medicine |
| Fac. Ingeniería | Facultad de Ingeniería / Faculty of Engineering |

## Scope
- **Date range**: 2020 – present
- **Work type**: journal articles
- **Attribution rule**: if a paper has *any* CEI author it is counted exclusively as CEI
  (no double-counting with other units)

## Data source
OpenAlex API (https://api.openalex.org) — no API key required.
Data retrieved via cursor pagination; cached locally to `openalex_cache.json`.


In [ ]:
# Install pyalex if not already present (required in Google Colab)
import importlib, subprocess, sys
if importlib.util.find_spec("pyalex") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyalex"])


In [ ]:
import re
import time
import json
import pathlib
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm
import pyalex
from pyalex import Works, config

# ── Polite pool — set your email here ────────────────────────────────────────
config.email = "castronallar@gmail.com"
config.max_retries = 5
config.retry_backoff_factor = 0.5
config.retry_http_codes = [429, 500, 503]

# ── Institution (ID resolved dynamically in next cell) ───────────────────────
UTALCA_ID = None   # set automatically by the institution lookup cell below
YEAR_MIN   = 2020

# ── CEI / CBSM detection patterns (research-center names are unique ─────────
# enough to imply UTalca affiliation regardless of how OpenAlex parsed
# the authorship's institutions array — so we scan ALL authorship raw
# strings for these, not just UTalca-tagged ones)
_CEI_PATTERNS = [
    r"centro\s+de\s+ecolog[íi]a\s+integrativa",
    r"center\s+for\s+integrative\s+ecology",
    r"\bCEI\b",
]
CEI_RE = re.compile("|".join(_CEI_PATTERNS), re.IGNORECASE)

_CBSM_PATTERNS = [
    r"centro\s+de\s+bioinform[áa]tica",
    r"center\s+for\s+bioinformatics",
    r"bioinformatics?,?\s+simulations?\s+and\s+modell?ing",
    r"\bCBSM\b",
]
CBSM_RE = re.compile("|".join(_CBSM_PATTERNS), re.IGNORECASE)

# ── Target unit patterns (order matters: first match wins) ───────────────────
# Centers / institutes come BEFORE the broad faculty patterns so a string
# like "Centro de Pomáceas, Facultad de Ciencias Agrarias, Universidad de
# Talca" is attributed to the center, not the faculty. CBSM is handled
# separately above (name-implies-UTalca rescue).
UNIT_LABEL_PATTERNS: list[tuple[re.Pattern, str]] = [
    (re.compile(r"pom[áa]ceas", re.I),                                                   "Centro Pomáceas"),
    (re.compile(r"vid\s+y\s+(el\s+)?vino|wine\s+and\s+vine", re.I),                 "CT Vid y Vino"),
    (re.compile(r"suelos\s+y\s+cultivos|soils\s+and\s+crops", re.I),                 "CT Suelos y Cultivos"),
    (re.compile(r"centro\s+tecnol[óo]gico\s+kipus|\bKipus\b", re.I),                 "Kipus"),
    (re.compile(r"plantas\s+nativas\s+de\s+chile|native\s+plants\s+of\s+chile", re.I),"Plantas Nativas Chile"),
    (re.compile(r"conversi[óo]n\s+de\s+energ[íi]a|energy\s+conversion", re.I),        "CT Conv. Energía"),
    (re.compile(r"transferencia\s+en\s+riego|riego\s+y\s+agroclimatolog[íi]a|\bCITRA\b", re.I),"CITRA"),
    (re.compile(r"estudios\s+constitucionales|\bCECOCH\b", re.I),                     "CECOCH"),
    (re.compile(r"mejoramiento\s+gen[ée]tico|fen[óo]mica\s+vegetal|plant\s+phenomics", re.I),"Mej. Genético"),
    (re.compile(r"centro\s+de\s+longevidad|longevity\s+center|\bvitalis\b", re.I),  "Vitalis"),
    (re.compile(r"psicolog[íi]a\s+aplicada|applied\s+psychology", re.I),               "C. Psic. Aplicada"),
    (re.compile(r"derecho\s+del\s+trabajo|labor\s+law|seguridad\s+social", re.I),    "C. Derecho Trabajo"),
    (re.compile(r"competitividad\s+del\s+maule|maule\s+competitiveness", re.I),       "Comp. Maule"),
    (re.compile(r"derechos.*(infancia|adolescencia)|infancia\s+y\s+adolescencia", re.I),"C. Infancia y Adolesc."),
    (re.compile(r"documentaci[óo]n\s+patrimonial|heritage\s+documentation", re.I),     "Documentación Patrimonial"),
    (re.compile(r"estudios\s+migratorios|\bCENEM\b", re.I),                           "CENEM"),
    (re.compile(r"ciencias\s+cognitivas|cognitive\s+sciences?\b", re.I),              "C. Ciencias Cognitivas"),
    (re.compile(r"centro\s+de\s+an[áa]lisis\s+pol[íi]tico|political\s+analysis\s+center", re.I),"C. Análisis Político"),
    (re.compile(r"centro\s+de\s+investigaci[óo]n\s+en\s+trombosis|trombosis\s+y\s+envejecimiento|thrombosis\s+and\s+(healthy\s+)?aging", re.I),"CITES"),
    (re.compile(r"derecho\s+penal|criminal\s+law", re.I),                              "C. Derecho Penal"),
    (re.compile(r"nanomedicina|nanomedicine", re.I),                                     "Nanomedicina"),
    (re.compile(r"derechos\s+de\s+las\s+minor[íi]as|minority\s+rights|gesti[óo]n\s+de\s+la\s+diversidad", re.I),"C. Minorías"),
    (re.compile(r"centro\s+internacional\s+de\s+educaci[óo]n\s+en\s+ingenier[íi]a|international\s+center\s+for\s+engineering\s+education", re.I),"CIEI"),
    # ── Faculties / institutes (fall through after the centers) ─────────
    (re.compile(r"ciencias\s+de\s+la\s+salud|health\s+sciences|faculty\s+of\s+health", re.I),
     "Fac. Ciencias de la Salud"),
    (re.compile(r"ciencias\s+biol[oó]gicas|biological\s+sciences|instituto\s+de\s+ciencias\s+biol", re.I),
     "Inst. Ciencias Biológicas"),
    (re.compile(r"ciencias\s+agrarias|agrarian\s+sciences|agronomy|agronom[ií]a|faculty\s+of\s+agri", re.I),
     "Fac. Ciencias Agrarias"),
    (re.compile(r"qu[íi]mica\s+de\s+recursos\s+naturales|natural\s+resources\s+chemistry|instituto\s+de\s+qu[íi]mica", re.I),
     "Inst. Química R. Naturales"),
    (re.compile(r"matem[aá]ticas|mathematics|instituto\s+de\s+matem", re.I),
     "Inst. Matemáticas"),
    (re.compile(r"medicina|medicine|escuela\s+de\s+medicina|faculty\s+of\s+medicine", re.I),
     "Fac. Medicina"),
    (re.compile(r"ingenier[íi]a|engineering|faculty\s+of\s+engineering", re.I),
     "Fac. Ingeniería"),
]

TARGET_UNITS = [label for _, label in UNIT_LABEL_PATTERNS]
ALL_UNITS    = ["CEI", "CBSM"] + TARGET_UNITS

# Display metadata per unit. Drives the side-panel tooltips, the tab
# grouping (centers vs faculties), and the bilingual labels on the
# dashboard. Keys must match the short labels used everywhere else.
UNIT_META: dict[str, dict] = {
    # ── Research / technological centers ───────────────────────────────
    "CEI":                     {"group": "centers", "es": "Centro de Ecología Integrativa",                          "en": "Center for Integrative Ecology"},
    "CBSM":                    {"group": "centers", "es": "Centro de Bioinformática, Simulación y Modelado",          "en": "Center for Bioinformatics, Simulations and Modelling"},
    "Centro Pomáceas":         {"group": "centers", "es": "Centro de Pomáceas",                                       "en": "Pomáceas Research Center"},
    "CT Vid y Vino":           {"group": "centers", "es": "Centro Tecnológico de la Vid y el Vino",                   "en": "Wine and Vine Technological Center"},
    "CT Suelos y Cultivos":    {"group": "centers", "es": "Centro Tecnológico de Suelos y Cultivos",                  "en": "Soils and Crops Technological Center"},
    "Kipus":                   {"group": "centers", "es": "Centro Tecnológico Kipus",                                 "en": "Kipus Technological Center"},
    "Plantas Nativas Chile":   {"group": "centers", "es": "Centro de Plantas Nativas de Chile",                       "en": "Center for Native Plants of Chile"},
    "CT Conv. Energía":        {"group": "centers", "es": "Centro Tecnológico de Conversión de Energía",              "en": "Energy Conversion Technological Center"},
    "CITRA":                   {"group": "centers", "es": "Centro de Investigación y Transferencia en Riego y Agroclimatología", "en": "Research and Transfer Center for Irrigation and Agroclimatology"},
    "CECOCH":                  {"group": "centers", "es": "Centro de Estudios Constitucionales de Chile",             "en": "Center for Constitutional Studies of Chile"},
    "Mej. Genético":           {"group": "centers", "es": "Centro de Mejoramiento Genético y Fenómica Vegetal",        "en": "Center for Plant Genetic Improvement and Phenomics"},
    "Vitalis":                 {"group": "centers", "es": "Centro de Longevidad Vitalis",                              "en": "Vitalis Longevity Center"},
    "C. Psic. Aplicada":       {"group": "centers", "es": "Centro de Psicología Aplicada",                            "en": "Center for Applied Psychology"},
    "C. Derecho Trabajo":      {"group": "centers", "es": "Centro de Estudios de Derecho del Trabajo y de la Seguridad Social", "en": "Center for Labor Law and Social Security Studies"},
    "Comp. Maule":             {"group": "centers", "es": "Centro de Competitividad del Maule",                       "en": "Maule Competitiveness Center"},
    "C. Infancia y Adolesc.":  {"group": "centers", "es": "Centro de Estudios sobre los Derechos de la Infancia y la Adolescencia", "en": "Center for Studies on Children and Adolescents' Rights"},
    "Documentación Patrimonial":{"group":"centers", "es": "Centro de Documentación Patrimonial",                      "en": "Heritage Documentation Center"},
    "CENEM":                   {"group": "centers", "es": "Centro Nacional de Estudios Migratorios",                  "en": "National Center for Migration Studies"},
    "C. Ciencias Cognitivas":  {"group": "centers", "es": "Centro de Investigación en Ciencias Cognitivas",           "en": "Center for Research in Cognitive Sciences"},
    "C. Análisis Político":    {"group": "centers", "es": "Centro de Análisis Político",                              "en": "Center for Political Analysis"},
    "CITES":                   {"group": "centers", "es": "Centro de Investigación en Trombosis y Envejecimiento Saludable", "en": "Center for Research on Thrombosis and Healthy Aging"},
    "C. Derecho Penal":        {"group": "centers", "es": "Centro de Estudios de Derecho Penal",                      "en": "Center for Criminal Law Studies"},
    "Nanomedicina":            {"group": "centers", "es": "Centro de Nanomedicina, Diagnóstico y Desarrollo de Fármacos", "en": "Center for Nanomedicine, Diagnostics and Drug Development"},
    "C. Minorías":             {"group": "centers", "es": "Centro de Derechos de las Minorías y Gestión de la Diversidad", "en": "Center for Minority Rights and Diversity Management"},
    "CIEI":                    {"group": "centers", "es": "Centro Internacional de Educación en Ingeniería",          "en": "International Center for Engineering Education"},
    # ── Faculties / institutes ─────────────────────────────────────────
    "Fac. Ciencias de la Salud":  {"group": "faculties", "es": "Facultad de Ciencias de la Salud",      "en": "Faculty of Health Sciences"},
    "Inst. Ciencias Biológicas":  {"group": "faculties", "es": "Instituto de Ciencias Biológicas",      "en": "Institute of Biological Sciences"},
    "Fac. Ciencias Agrarias":     {"group": "faculties", "es": "Facultad de Ciencias Agrarias",         "en": "Faculty of Agrarian Sciences"},
    "Inst. Química R. Naturales": {"group": "faculties", "es": "Instituto de Química de Recursos Naturales", "en": "Institute of Natural Resources Chemistry"},
    "Inst. Matemáticas":          {"group": "faculties", "es": "Instituto de Matemáticas",              "en": "Institute of Mathematics"},
    "Fac. Medicina":              {"group": "faculties", "es": "Facultad de Medicina",                  "en": "Faculty of Medicine"},
    "Fac. Ingeniería":            {"group": "faculties", "es": "Facultad de Ingeniería",                "en": "Faculty of Engineering"},
}

print("Configuration loaded.")
print(f"Tracking {sum(1 for u in UNIT_META.values() if u['group']=='centers')} centers + {sum(1 for u in UNIT_META.values() if u['group']=='faculties')} faculties/institutes, articles from {YEAR_MIN}+")


In [ ]:
CACHE_FILE = pathlib.Path("openalex_cache.json")
SOURCES_CACHE_FILE = pathlib.Path("sources_cache.json")

def save_cache(works: list, path: pathlib.Path = CACHE_FILE) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(works, f)
    print(f"Cached {len(works):,} works → {path}")

def load_cache(path: pathlib.Path = CACHE_FILE) -> list | None:
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"Loaded {len(data):,} works from cache ({path})")
        return data
    return None

def save_sources_cache(sources: dict, path: pathlib.Path = SOURCES_CACHE_FILE) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(sources, f)
    print(f"Cached {len(sources):,} sources → {path}")

def load_sources_cache(path: pathlib.Path = SOURCES_CACHE_FILE) -> dict | None:
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"Loaded {len(data):,} sources from cache ({path})")
        return data
    return None


In [ ]:
from pyalex import Institutions

results = Institutions().search("Universidad de Talca").get()
print("Matches for 'Universidad de Talca' in OpenAlex:")
for r in results:
    print(f"  {r['id']}  |  {r['display_name']}  |  ROR: {r.get('ror','—')}  |  Works: {r.get('works_count',0):,}")

# Auto-select the first match if only one result
if len(results) == 1:
    UTALCA_ID = results[0]["id"]
    print(f"\nUsing: {UTALCA_ID}")
else:
    # If multiple hits, pick the one with the most works (most likely the right one)
    best = max(results, key=lambda r: r.get("works_count", 0))
    UTALCA_ID = best["id"]
    print(f"\nAuto-selected best match: {UTALCA_ID}  ({best['display_name']})")


In [ ]:
def fetch_utalca_works() -> list:
    query = (
        Works()
        .filter(authorships={"institutions": {"id": UTALCA_ID}})
        .filter(type="article")
        .filter(publication_year=f">{YEAR_MIN - 1}")
    )
    all_works = []
    pages = query.paginate(method="cursor", per_page=200, n_max=50_000)
    for page in tqdm(pages, desc="Fetching from OpenAlex"):
        all_works.extend(list(page))
        time.sleep(0.12)
    return all_works

raw_works = load_cache()
if not raw_works:
    raw_works = fetch_utalca_works()
    save_cache(raw_works)

print(f"Total works: {len(raw_works):,}")


### Journal-source enrichment

For each unique primary-source (journal) referenced by our works, fetch the
source's `summary_stats["2yr_mean_citedness"]` from the OpenAlex *Sources*
endpoint. This is OpenAlex's open analog of a 2-year Journal Impact Factor
(it is **not** Clarivate's JCR IF, which is paywalled). The values are
cached to `sources_cache.json` so re-runs are instant.


In [ ]:
from pyalex import Sources

def collect_source_ids(works: list) -> list[str]:
    ids: set[str] = set()
    for w in works:
        loc = w.get("primary_location") or {}
        src = loc.get("source") or {}
        sid = src.get("id")
        if sid:
            ids.add(sid)
    return sorted(ids)

def _short_id(full_id: str) -> str:
    return full_id.rsplit("/", 1)[-1]

def fetch_sources(source_ids: list[str]) -> dict:
    """Fetch sources in batches of 50 via OpenAlex; return {id: source_dict}.

    Uses the openalex_id filter with pipe-OR (per OpenAlex docs). Falls back
    to per-id fetch on batch errors so a single bad ID doesn't break a run.
    """
    out: dict[str, dict] = {}
    batch_size = 50
    batches = [source_ids[i:i+batch_size] for i in range(0, len(source_ids), batch_size)]
    for batch in tqdm(batches, desc="Fetching sources from OpenAlex"):
        short_ids = [_short_id(s) for s in batch]
        try:
            results = Sources().filter(openalex_id="|".join(short_ids)).get(per_page=batch_size)
            got = {s["id"]: s for s in results if s.get("id")}
            # If the filter returned fewer than requested (e.g. some IDs invalid),
            # fall back to per-id for the missing ones.
            missing = [sid for sid in batch if sid not in got]
            for sid in missing:
                try:
                    out[sid] = Sources()[_short_id(sid)]
                except Exception:
                    pass
                time.sleep(0.05)
            out.update(got)
        except Exception as e:
            print(f"  ! batch error ({len(batch)} ids), falling back per-id: {e}")
            for sid in batch:
                try:
                    out[sid] = Sources()[_short_id(sid)]
                except Exception:
                    pass
                time.sleep(0.05)
        time.sleep(0.12)
    return out

sources_cache = load_sources_cache() or {}
needed_ids = [sid for sid in collect_source_ids(raw_works) if sid not in sources_cache]
if needed_ids:
    print(f"Fetching {len(needed_ids):,} new sources (cached: {len(sources_cache):,})")
    fresh = fetch_sources(needed_ids)
    sources_cache.update(fresh)
    save_sources_cache(sources_cache)
else:
    print(f"All {len(sources_cache):,} sources already cached")

# Build lookups from the source cache
JOURNAL_2YR_IF: dict[str, float] = {}
JOURNAL_PUBLISHER: dict[str, str] = {}
for sid, src in sources_cache.items():
    stats = src.get("summary_stats") or {}
    val = stats.get("2yr_mean_citedness")
    if val is not None:
        JOURNAL_2YR_IF[sid] = float(val)
    pub = src.get("host_organization_name")
    if pub:
        JOURNAL_PUBLISHER[sid] = pub

print(f"Journals with 2-year mean citedness: {len(JOURNAL_2YR_IF):,} / {len(sources_cache):,}")
print(f"Journals with publisher name:        {len(JOURNAL_PUBLISHER):,} / {len(sources_cache):,}")


### WoS / Scopus enrichment

Cross-reference our journals against `Revistas por cuartil SJR-JCR 2023.xlsx`
to attach the Web of Science **JCR Impact Factor + best quartile** and Scopus
**SJR + best quartile** to every paper, joined by ISSN first and then by
normalized journal name as a fallback.

Both sheets share the same overall layout — header on row 18, multi-ISSN
strings like `"00079235, 15424863"` per row. The lookup is built once per
source (journal) and then attached to each paper in `build_records`.


In [ ]:
import re
import urllib.request

WOS_SCOPUS_FILE = pathlib.Path("Revistas por cuartil SJR-JCR 2023.xlsx")
WOS_SCOPUS_URL  = (
    "https://raw.githubusercontent.com/ecastron/cei_biblio/"
    "claude/project-planning-PKOFJ/"
    "Revistas%20por%20cuartil%20SJR-JCR%202023.xlsx"
)

if not WOS_SCOPUS_FILE.exists():
    print(f"Downloading {WOS_SCOPUS_FILE.name} from GitHub …")
    try:
        urllib.request.urlretrieve(WOS_SCOPUS_URL, WOS_SCOPUS_FILE)
        print(f"  ✓ saved → {WOS_SCOPUS_FILE} ({WOS_SCOPUS_FILE.stat().st_size/1024/1024:.1f} MB)")
    except Exception as e:
        print(f"  ✗ download failed: {e}")
        print("  WoS / Scopus columns will be null in the export.")

def _norm_issn(s: str) -> str | None:
    if not s: return None
    s = re.sub(r"[^0-9Xx]", "", str(s)).upper()
    return s if len(s) == 8 else None

def _norm_name(s: str) -> str | None:
    if not s or pd.isna(s): return None
    s = str(s).lower()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

def _norm_quartile(q):
    if q is None or (isinstance(q, float) and pd.isna(q)): return None
    s = str(q).strip().upper()
    if s in {"Q1", "Q2", "Q3", "Q4"}: return s
    return None

def _norm_value(v):
    try:
        f = float(v)
        return f if not pd.isna(f) else None
    except (TypeError, ValueError):
        return None

def _split_issns(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)): return []
    out = []
    for tok in re.split(r"[,;\s]+", str(raw)):
        n = _norm_issn(tok)
        if n: out.append(n)
    return out

def _load_metric_sheet(file: pathlib.Path, sheet: str, value_col: str) -> tuple[dict, dict]:
    """Return (issn_lookup, name_lookup) for one sheet."""
    if not file.exists():
        print(f"  ! {file} not found — skipping {sheet}")
        return {}, {}
    df = pd.read_excel(file, sheet_name=sheet, header=18).dropna(how="all")
    issn_lookup: dict[str, dict] = {}
    name_lookup: dict[str, dict] = {}
    for _, row in df.iterrows():
        rec = {
            "value":    _norm_value(row.get(value_col)),
            "quartile": _norm_quartile(row.get("Mejor Q")),
        }
        if rec["value"] is None and rec["quartile"] is None:
            continue
        for issn in _split_issns(row.get("ISSN")):
            issn_lookup.setdefault(issn, rec)
        n = _norm_name(row.get("Revista"))
        if n:
            name_lookup.setdefault(n, rec)
    print(f"  {sheet}: {len(df):,} rows → {len(issn_lookup):,} ISSNs · {len(name_lookup):,} names")
    return issn_lookup, name_lookup

wos_issn,    wos_name    = _load_metric_sheet(WOS_SCOPUS_FILE, "WoS",    "JCR")
scopus_issn, scopus_name = _load_metric_sheet(WOS_SCOPUS_FILE, "Scopus", "SJR")

def _source_issns(src: dict) -> list[str]:
    out: list[str] = []
    if src.get("issn_l"):
        n = _norm_issn(src["issn_l"])
        if n: out.append(n)
    for raw in (src.get("issn") or []):
        n = _norm_issn(raw)
        if n and n not in out: out.append(n)
    return out

def _match(src: dict, issn_lookup: dict, name_lookup: dict) -> dict:
    for issn in _source_issns(src):
        if issn in issn_lookup:
            return issn_lookup[issn]
    n = _norm_name(src.get("display_name"))
    if n and n in name_lookup:
        return name_lookup[n]
    return {"value": None, "quartile": None}

JOURNAL_WOS:    dict[str, dict] = {}
JOURNAL_SCOPUS: dict[str, dict] = {}
n_wos_hit = n_scopus_hit = 0
for sid, src in sources_cache.items():
    w = _match(src, wos_issn, wos_name)
    s = _match(src, scopus_issn, scopus_name)
    JOURNAL_WOS[sid]    = w
    JOURNAL_SCOPUS[sid] = s
    if w["value"] is not None or w["quartile"]: n_wos_hit += 1
    if s["value"] is not None or s["quartile"]: n_scopus_hit += 1

print(f"\nMatch coverage across {len(sources_cache):,} cached sources:")
print(f"  WoS    : {n_wos_hit:,} ({100*n_wos_hit/max(len(sources_cache),1):.1f}%)")
print(f"  Scopus : {n_scopus_hit:,} ({100*n_scopus_hit/max(len(sources_cache),1):.1f}%)")

# ── Manual overrides (journal_overrides.csv) ────────────────────────────
# Fill in or correct individual journals not covered by the main xlsx.
OVERRIDES_FILE = pathlib.Path("journal_overrides.csv")
OVERRIDES_URL  = (
    "https://raw.githubusercontent.com/ecastron/cei_biblio/"
    "claude/project-planning-PKOFJ/journal_overrides.csv"
)
if not OVERRIDES_FILE.exists():
    try:
        urllib.request.urlretrieve(OVERRIDES_URL, OVERRIDES_FILE)
    except Exception as e:
        print(f"  (no journal_overrides.csv: {e})")

if OVERRIDES_FILE.exists():
    ov = pd.read_csv(OVERRIDES_FILE).fillna("")
    n_applied_w = n_applied_s = n_skipped = 0
    for _, row in ov.iterrows():
        rec_w = {
            "value":    _norm_value(row.get("wos_jif")),
            "quartile": _norm_quartile(row.get("wos_quartile")),
        }
        rec_s = {
            "value":    _norm_value(row.get("scopus_sjr")),
            "quartile": _norm_quartile(row.get("scopus_quartile")),
        }
        if rec_w["value"] is None and rec_w["quartile"] is None \
           and rec_s["value"] is None and rec_s["quartile"] is None:
            continue
        # Find matching source(s) — try ISSN first, then normalized name
        target_sids: list[str] = []
        issns = _split_issns(row.get("issn"))
        nm    = _norm_name(row.get("journal"))
        for sid, src in sources_cache.items():
            if issns and any(i in _source_issns(src) for i in issns):
                target_sids.append(sid); continue
            if nm and _norm_name(src.get("display_name")) == nm:
                target_sids.append(sid)
        if not target_sids:
            n_skipped += 1
            print(f"  ! override unmatched: {row.get('journal', '?')!r} (issn={row.get('issn','')!r})")
            continue
        for sid in target_sids:
            if rec_w["value"] is not None or rec_w["quartile"]:
                cur = JOURNAL_WOS.get(sid, {"value": None, "quartile": None}).copy()
                if rec_w["value"]    is not None: cur["value"]    = rec_w["value"]
                if rec_w["quartile"]:             cur["quartile"] = rec_w["quartile"]
                JOURNAL_WOS[sid] = cur
                n_applied_w += 1
            if rec_s["value"] is not None or rec_s["quartile"]:
                cur = JOURNAL_SCOPUS.get(sid, {"value": None, "quartile": None}).copy()
                if rec_s["value"]    is not None: cur["value"]    = rec_s["value"]
                if rec_s["quartile"]:             cur["quartile"] = rec_s["quartile"]
                JOURNAL_SCOPUS[sid] = cur
                n_applied_s += 1
    print(f"\nManual overrides applied: WoS {n_applied_w} · Scopus {n_applied_s} · unmatched {n_skipped}")


In [ ]:
def normalize_unit(raw: str) -> str:
    for pattern, label in UNIT_LABEL_PATTERNS:
        if pattern.search(raw):
            return label
    return "Other UTalca"

def classify_work(work: dict) -> set:
    """Return the set of unit labels for this work.

    CEI takes exclusive ownership. CEI / CBSM are detected on ANY
    authorship's raw affiliation strings (these names imply UTalca
    even if OpenAlex parsed the institution as a separate entity).
    Other faculties / institutes are only credited when the authorship
    is parsed as UTalca-affiliated, to avoid attributing a foreign
    author's "Faculty of Engineering" to UTalca's.
    """
    all_raws = [
        raw
        for a in work.get("authorships", [])
        for raw in a.get("raw_affiliation_strings", [])
    ]
    if any(CEI_RE.search(r) for r in all_raws):
        return {"CEI"}

    units: set[str] = set()
    if any(CBSM_RE.search(r) for r in all_raws):
        units.add("CBSM")

    for authorship in work.get("authorships", []):
        inst_ids = [i.get("id", "") for i in authorship.get("institutions", [])]
        if not any(UTALCA_ID in iid for iid in inst_ids):
            continue
        for raw in authorship.get("raw_affiliation_strings", []):
            unit = normalize_unit(raw)
            if unit != "Other UTalca":
                units.add(unit)
    return units or {"Other UTalca"}


In [ ]:
def _extract_journal(work: dict) -> tuple[str, str | None]:
    """Pull a journal display name + OpenAlex source id from a work record.

    OpenAlex sometimes leaves ``primary_location.source.display_name``
    empty even when the same name is available under the legacy
    ``host_venue`` field or under another entry in ``locations[]``.
    Walk the fallback chain so engineering / proceedings articles
    don't all collapse into "Unknown Journal".
    """
    loc = work.get("primary_location") or {}
    src = loc.get("source") or {}
    if src.get("display_name"):
        return src["display_name"], src.get("id")

    hv = work.get("host_venue") or {}
    if hv.get("display_name"):
        return hv["display_name"], hv.get("id")

    for other in (work.get("locations") or []):
        osrc = other.get("source") or {}
        if osrc.get("display_name"):
            return osrc["display_name"], osrc.get("id")

    return "Unknown Journal", src.get("id")  # may still be None

def build_records(works: list) -> pd.DataFrame:
    rows = []
    _empty = {"value": None, "quartile": None}
    for work in works:
        unit_set = classify_work(work)
        journal, journal_id = _extract_journal(work)
        journal_2yr_if = JOURNAL_2YR_IF.get(journal_id) if journal_id else None
        publisher      = JOURNAL_PUBLISHER.get(journal_id) if journal_id else None
        wos    = JOURNAL_WOS.get(journal_id, _empty)    if journal_id else _empty
        scopus = JOURNAL_SCOPUS.get(journal_id, _empty) if journal_id else _empty
        base = {
            "openalex_id":              work.get("id", ""),
            "doi":                      work.get("doi", ""),
            "title":                    work.get("title", ""),
            "year":                     work.get("publication_year"),
            "cited_by_count":           work.get("cited_by_count", 0) or 0,
            "journal":                  journal,
            "journal_id":               journal_id,
            "journal_2yr_mean_citedness": journal_2yr_if,
            "publisher":                publisher,
            "wos_jif":                  wos["value"],
            "wos_quartile":             wos["quartile"],
            "scopus_sjr":               scopus["value"],
            "scopus_quartile":          scopus["quartile"],
        }
        for unit in unit_set:
            rows.append({**base, "unit": unit})
    df = pd.DataFrame(rows)
    df["year"]           = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["cited_by_count"] = pd.to_numeric(df["cited_by_count"], errors="coerce").fillna(0)
    df["journal_2yr_mean_citedness"] = pd.to_numeric(
        df["journal_2yr_mean_citedness"], errors="coerce"
    )
    return df

df = build_records(raw_works)
print(f"Rows (work×unit): {len(df):,}  |  Unique works: {df['openalex_id'].nunique():,}")
print("\nPapers per unit:")
print(df.groupby('unit')['openalex_id'].nunique().sort_values(ascending=False).to_string())


In [ ]:
assert df["openalex_id"].notna().all(), "Null OpenAlex IDs found"
assert (df["cited_by_count"] >= 0).all(), "Negative citation counts"

cei_n = df[df["unit"] == "CEI"]["openalex_id"].nunique()
assert cei_n > 0, "CEI matched 0 papers — check patterns or institution ID"
print(f"CEI papers : {cei_n:,}")
print(f"Year range : {df['year'].min()} – {df['year'].max()}")
print(f"Null years : {df['year'].isna().sum()}")

# Sample unclassified raw strings for pattern tuning
other_works = [w for w in raw_works if classify_work(w) == {"Other UTalca"}]
sample_raws = []
for w in other_works[:40]:
    for a in w.get("authorships", []):
        if any(UTALCA_ID in i.get("id","") for i in a.get("institutions",[])):
            sample_raws.extend(a.get("raw_affiliation_strings", []))
print(f"\n'Other UTalca' papers: {len(other_works):,}")
print("Sample unclassified raw strings (first 10 unique):")
for s in list(dict.fromkeys(sample_raws))[:10]:
    print(" •", s)


## Analysis 1 — Publication Counts Over Time

In [ ]:
FOCUS_UNITS = ALL_UNITS   # CEI + 7 named units

pub = (
    df[df["unit"].isin(FOCUS_UNITS)]
    .groupby(["unit", "year"])["openalex_id"]
    .nunique()
    .rename("count")
    .reset_index()
)

years = sorted(df["year"].dropna().unique())
pivot = (
    pub.pivot(index="year", columns="unit", values="count")
    .reindex(years)
    .fillna(0)
)

palette = {
    "CEI":                       "#1f77b4",
    "CBSM":                      "#17becf",
    "Fac. Ciencias de la Salud": "#ff7f0e",
    "Inst. Ciencias Biológicas": "#2ca02c",
    "Fac. Ciencias Agrarias":    "#d62728",
    "Inst. Química R. Naturales":"#9467bd",
    "Inst. Matemáticas":         "#8c564b",
    "Fac. Medicina":             "#e377c2",
    "Fac. Ingeniería":           "#bcbd22",
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: line chart — CEI bold, others thinner
ax1 = axes[0]
for unit in FOCUS_UNITS:
    if unit not in pivot.columns:
        continue
    lw   = 2.8 if unit == "CEI" else 1.4
    ms   = 5   if unit == "CEI" else 3
    zord = 5   if unit == "CEI" else 2
    ax1.plot(pivot.index, pivot[unit],
             label=unit, color=palette.get(unit, "gray"),
             linewidth=lw, marker="o", markersize=ms, zorder=zord)
ax1.set_title("Annual Publications — CEI vs Academic Units (2020+)")
ax1.set_xlabel("Year")
ax1.set_ylabel("Publications")
ax1.legend(fontsize=8, loc="upper left")
ax1.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Right: grouped bar chart per year
ax2 = axes[1]
bar_data = pivot[[u for u in FOCUS_UNITS if u in pivot.columns]]
bar_data.plot(kind="bar", ax=ax2, color=[palette.get(u,"gray") for u in bar_data.columns],
              width=0.75, edgecolor="white")
ax2.set_title("Annual Publications — Grouped Bar")
ax2.set_xlabel("Year")
ax2.set_ylabel("Publications")
ax2.legend(fontsize=7, loc="upper left")
ax2.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("fig1_publication_counts.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig1_publication_counts.png")


## Analysis 2 — Citation Impact

In [ ]:
def h_index(series: pd.Series) -> int:
    counts = sorted(series.dropna().astype(int), reverse=True)
    h = 0
    for i, c in enumerate(counts, 1):
        if c >= i:
            h = i
        else:
            break
    return h

# Deduplicate per work×unit before aggregating citations
work_unit = df[df["unit"].isin(FOCUS_UNITS)].drop_duplicates(subset=["openalex_id","unit"])

impact = (
    work_unit.groupby("unit")["cited_by_count"]
    .agg(
        total_citations="sum",
        mean_citations="mean",
        median_citations="median",
        paper_count="count",
        h_index=h_index,
    )
    .reset_index()
    .sort_values("h_index", ascending=False)
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = [
    ("h_index",        "h-index",               "h-index"),
    ("mean_citations", "Mean Citations / Paper", "Mean citations"),
    ("total_citations","Total Citations",         "Total citations"),
]
for ax, (col, title, xlabel) in zip(axes, metrics):
    colors = [palette.get(u, "#aec7e8") for u in impact["unit"]]
    ax.barh(impact["unit"], impact[col], color=colors, edgecolor="white")
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    # Highlight CEI bar edge
    cei_idx = impact[impact["unit"]=="CEI"].index
    if len(cei_idx):
        ax.patches[int(cei_idx[0])].set_edgecolor("black")
        ax.patches[int(cei_idx[0])].set_linewidth(1.5)

plt.tight_layout()
plt.savefig("fig2_citation_impact.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig2_citation_impact.png")
display(
    impact.style
    .highlight_max(subset=["total_citations","mean_citations","h_index"], color="#cfe2ff")
    .format({"mean_citations": "{:.2f}", "median_citations": "{:.1f}",
             "total_citations": "{:,.0f}"})
    .set_caption("Citation Impact by Unit")
)


## Analysis 3 — Journal Distribution

In [ ]:
TOP_N = 10
journals_df = df[df["unit"].isin(FOCUS_UNITS)].drop_duplicates(subset=["openalex_id","unit"])

def top_journals(unit_name: str, n: int = TOP_N) -> pd.DataFrame:
    return (
        journals_df[journals_df["unit"] == unit_name]
        .groupby("journal")["openalex_id"].count()
        .nlargest(n)
        .rename("count")
        .reset_index()
    )

# ── Fig 3a: CEI top journals ──────────────────────────────────────────────────
cei_j = top_journals("CEI")
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(cei_j["journal"][::-1], cei_j["count"][::-1], color="#1f77b4")
ax.set_title(f"Top {TOP_N} Journals — CEI")
ax.set_xlabel("Publications")
ax.set_yticklabels(
    [j if len(j) < 45 else j[:42]+"…" for j in cei_j["journal"][::-1]], fontsize=9
)
plt.tight_layout()
plt.savefig("fig3a_cei_journals.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Fig 3b: top journals per comparison unit (small multiples) ────────────────
n_cols = 2
n_rows = (len(TARGET_UNITS) + 1) // 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes_flat = axes.flatten()

for i, unit in enumerate(TARGET_UNITS):
    ax = axes_flat[i]
    jdf = top_journals(unit, TOP_N)
    if jdf.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title(unit)
        continue
    ax.barh(jdf["journal"][::-1], jdf["count"][::-1], color=palette.get(unit,"#aec7e8"))
    ax.set_title(f"Top {TOP_N} Journals — {unit}", fontsize=9)
    ax.set_xlabel("Publications", fontsize=8)
    ax.set_yticklabels(
        [j if len(j) < 38 else j[:35]+"…" for j in jdf["journal"][::-1]], fontsize=7
    )

for j in range(i+1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle("Top Journals by Academic Unit", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("fig3b_unit_journals.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Fig 4: heatmap — units × CEI's core journals ─────────────────────────────
cei_core = set(
    journals_df[journals_df["unit"]=="CEI"]
    .groupby("journal")["openalex_id"].count()
    .pipe(lambda s: s[s >= 2]).index
)

shared = journals_df[journals_df["journal"].isin(cei_core)]
hmap = (
    shared.groupby(["unit","journal"])["openalex_id"]
    .count()
    .unstack(fill_value=0)
)

if not hmap.empty:
    fig, ax = plt.subplots(figsize=(max(12, len(hmap.columns)*0.55), max(4, len(hmap)*0.5)))
    sns.heatmap(hmap, annot=True, fmt="d", cmap="YlOrRd",
                linewidths=0.4, linecolor="white", ax=ax)
    ax.set_title("Publications per Unit in CEI's Core Journals (≥2 CEI papers)")
    ax.set_xlabel("Journal")
    ax.set_ylabel("")
    plt.xticks(rotation=40, ha="right", fontsize=7)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.savefig("fig4_journal_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved fig4_journal_heatmap.png")
else:
    print("Heatmap skipped — no shared journals found.")
print("Saved fig3a_cei_journals.png  fig3b_unit_journals.png")


## Summary Table

In [ ]:
summary = (
    work_unit.groupby("unit")
    .agg(
        papers       = ("openalex_id", "nunique"),
        total_cit    = ("cited_by_count", "sum"),
        mean_cit     = ("cited_by_count", "mean"),
        h_index      = ("cited_by_count", h_index),
        mean_if      = ("journal_2yr_mean_citedness", "mean"),
        median_if    = ("journal_2yr_mean_citedness", "median"),
        first_year   = ("year", "min"),
        last_year    = ("year", "max"),
    )
    .sort_values("papers", ascending=False)
    .reset_index()
    .rename(columns={"unit":"Unit","papers":"Papers",
                     "total_cit":"Total Citations","mean_cit":"Mean Cit./Paper",
                     "h_index":"h-index",
                     "mean_if":"Mean Journal IF","median_if":"Median Journal IF",
                     "first_year":"From","last_year":"To"})
)

display(
    summary.style
    .background_gradient(subset=["Papers","Total Citations","h-index"], cmap="Blues")
    .format({
        "Mean Cit./Paper":   "{:.2f}",
        "Total Citations":   "{:,.0f}",
        "Mean Journal IF":   "{:.2f}",
        "Median Journal IF": "{:.2f}",
    }, na_rep="—")
    .set_caption("Bibliometric Summary — Universidad de Talca (2020+)")
)


In [ ]:
import datetime

# ── CSV export ────────────────────────────────────────────────────────────────
df.to_csv("works_by_unit.csv", index=False)
summary.to_csv("summary_by_unit.csv", index=False)
print("Exported works_by_unit.csv and summary_by_unit.csv")

# ── Website data export ───────────────────────────────────────────────────────
docs_dir = pathlib.Path("docs")
docs_dir.mkdir(exist_ok=True)

work_unit_export = df[df["unit"].isin(ALL_UNITS)].drop_duplicates(subset=["openalex_id", "unit"])

papers_export = [
    {
        "id":             row["openalex_id"],
        "unit":           row["unit"],
        "year":           int(row["year"]) if pd.notna(row["year"]) else None,
        "journal":        row["journal"],
        "journal_id":     row["journal_id"] if pd.notna(row["journal_id"]) else None,
        "journal_2yr_mean_citedness": (
            round(float(row["journal_2yr_mean_citedness"]), 3)
            if pd.notna(row["journal_2yr_mean_citedness"]) else None
        ),
        "publisher":      row["publisher"]      if pd.notna(row["publisher"])      else None,
        "wos_jif":        (round(float(row["wos_jif"]),   3) if pd.notna(row["wos_jif"])    else None),
        "wos_quartile":   row["wos_quartile"]   if pd.notna(row["wos_quartile"])   else None,
        "scopus_sjr":     (round(float(row["scopus_sjr"]), 3) if pd.notna(row["scopus_sjr"]) else None),
        "scopus_quartile":row["scopus_quartile"] if pd.notna(row["scopus_quartile"]) else None,
        "title":          row["title"],
        "doi":            row["doi"] if pd.notna(row["doi"]) and row["doi"] else None,
        "cited_by_count": int(row["cited_by_count"]),
    }
    for _, row in work_unit_export.iterrows()
]

summary_export = [
    {
        "unit":                       row["Unit"],
        "papers":                     int(row["Papers"]),
        "total_citations":            int(row["Total Citations"]),
        "mean_citations":             round(float(row["Mean Cit./Paper"]), 2),
        "h_index":                    int(row["h-index"]),
        "mean_journal_2yr_citedness": (
            round(float(row["Mean Journal IF"]), 2)
            if pd.notna(row["Mean Journal IF"]) else None
        ),
        "median_journal_2yr_citedness": (
            round(float(row["Median Journal IF"]), 2)
            if pd.notna(row["Median Journal IF"]) else None
        ),
        "first_year":      int(row["From"]) if pd.notna(row["From"]) else None,
        "last_year":       int(row["To"])   if pd.notna(row["To"])   else None,
    }
    for _, row in summary.iterrows()
]

# Only export units that actually have papers in the filtered set
active_units = [u for u in ALL_UNITS if u in set(work_unit_export["unit"].unique())]
# Always emit metadata for active units; the dashboard reads it for
# tab grouping and bilingual tooltips.
unit_meta_export = {u: UNIT_META[u] for u in active_units if u in UNIT_META}

site_data = {
    "generated":  datetime.date.today().isoformat(),
    "year_range": [YEAR_MIN, int(df["year"].dropna().max())],
    "units":      active_units,
    "unit_meta":  unit_meta_export,
    "papers":     papers_export,
    "summary":    summary_export,
}

with open(docs_dir / "data.json", "w", encoding="utf-8") as fh:
    json.dump(site_data, fh, ensure_ascii=False, indent=2)

print(f"Exported {len(papers_export):,} paper records → docs/data.json")
print(f"Active units (with papers): {len(active_units)} "
      f"({sum(1 for u in active_units if UNIT_META.get(u,{}).get('group')=='centers')} centers + "
      f"{sum(1 for u in active_units if UNIT_META.get(u,{}).get('group')=='faculties')} faculties/institutes)")
print("Interactive website ready — open docs/index.html or enable GitHub Pages on the docs/ folder")
